In [26]:
import os
from google.colab import drive
import subprocess
import time
import re
import threading
import io

# 1. Montar Drive
drive.mount('/content/drive')

# 2. Rutas persistentes
CARPETA_DRIVE = "/content/drive/MyDrive/Colab Notebooks/llm_setup"
CARPETA_BIN = f"{CARPETA_DRIVE}/bin"
MODELO_NOMBRE = "qwen2.5-7b-instruct-q4_k_m.gguf"
MODEL_PATH = f"{CARPETA_DRIVE}/{MODELO_NOMBRE}"
URL_DESCARGA = "https://huggingface.co/bartowski/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf"

# Crear carpetas si no existen
os.makedirs(CARPETA_BIN, exist_ok=True)

# 3. Descargar Modelo si no existe
if not os.path.exists(MODEL_PATH):
    print("Descargando modelo a Drive...")
    !wget -O "{MODEL_PATH}" {URL_DESCARGA}
else:
    print("Modelo ya presente en Drive.")

# Función de ayuda para leer la salida de un stream en un hilo
def read_stream_in_thread(stream, buffer, name):
    for line in iter(stream.readline, b''):
        decoded_line = line.decode('utf-8', errors='ignore').strip()
        if decoded_line:
            buffer.write(decoded_line + '\n')
            print(f"[{name}] {decoded_line}", flush=True) # Imprimir en tiempo real

# Función para terminar procesos existentes por nombre
def kill_process(name):
    pids = subprocess.run(['pgrep', '-f', name], capture_output=True, text=True).stdout.strip().split('\n')
    for pid in pids:
        if pid.isdigit():
            print(f"Terminando proceso '{name}' con PID {pid}...")
            subprocess.run(['kill', '-9', pid])
            time.sleep(1) # Dar tiempo para que el proceso termine

# 4. Compilar O RECUPERAR el ejecutable
llama_server_local_path = "/content/llama-server"

# Asegurarse de que cualquier instancia de llama-server esté terminada antes de copiar el ejecutable
kill_process(llama_server_local_path)

if not os.path.exists(f"{CARPETA_BIN}/llama-server"):
    print("Compilando llama.cpp (esto solo pasará una vez)...", flush=True)
    !git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
    !cmake -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -S /content/llama.cpp
    !cmake --build /content/llama.cpp/build --config Release -j $(nproc)
    !cp /content/llama.cpp/build/bin/llama-server "{CARPETA_BIN}/llama-server"
    print("Binario guardado en Drive.", flush=True)
    !cp "{CARPETA_BIN}/llama-server" {llama_server_local_path}
    !chmod +x {llama_server_local_path}
else:
    print("Ejecutable encontrado en Drive. Copiando a entorno local...", flush=True)
    !cp "{CARPETA_BIN}/llama-server" {llama_server_local_path}
    !chmod +x {llama_server_local_path}


# 5. Lanzar servidor en background y esperar a que esté listo
print("\n--- Iniciando Servidor Llama.cpp ---", flush=True)

# Terminar cualquier instancia anterior de llama-server si aún existiera (doble chequeo)
kill_process(llama_server_local_path)

# Buffers para capturar la salida de llama-server
llama_stdout_buffer = io.StringIO()
llama_stderr_buffer = io.StringIO()

print(f"Iniciando llama-server directamente con subprocess.Popen.", flush=True)

llama_process = subprocess.Popen(
    [llama_server_local_path, '-m', MODEL_PATH, '-ngl', '99', '--host', '0.0.0.0', '--port', '8889', '--temp', '0.1'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    bufsize=1,
    universal_newlines=False
)

# Hilos para leer stdout y stderr de llama-server
llama_stdout_thread = threading.Thread(target=read_stream_in_thread, args=(llama_process.stdout, llama_stdout_buffer, 'LLAMA_STDOUT'))
llama_stderr_thread = threading.Thread(target=read_stream_in_thread, args=(llama_process.stderr, llama_stderr_buffer, 'LLAMA_STDERR'))
llama_stdout_thread.daemon = True
llama_stderr_thread.daemon = True
llama_stdout_thread.start()
llama_stderr_thread.start()

server_ready = False
server_timeout = 180 # Aumentado el tiempo de espera para el inicio del servidor (ej. 3 minutos para la carga de modelos grandes)
check_interval = 2
server_start_time = time.time()

print(f"Esperando a que el servidor llama.cpp esté listo (hasta {server_timeout} segundos)...")
while time.time() - server_start_time < server_timeout:
    current_llama_stdout_full = llama_stdout_buffer.getvalue()
    current_llama_stderr_full = llama_stderr_buffer.getvalue() # También revisar stderr

    if "listening on http://0.0.0.0:8889" in current_llama_stdout_full or \
       "listening on http://0.0.0.0:8889" in current_llama_stderr_full:
        server_ready = True
        print("\nServidor llama.cpp listo y escuchando en el puerto 8889.", flush=True)
        break

    if llama_process.poll() is not None: # El proceso ha terminado prematuramente
        print(f"\nERROR: El proceso de llama-server terminó inesperadamente con código de salida {llama_process.returncode}.", flush=True)
        print("\n--- Salida Final de LLAMA.cpp STDOUT ---\n" + llama_stdout_buffer.getvalue(), flush=True)
        print("\n--- Salida Final de LLAMA.cpp STDERR ---\n" + llama_stderr_buffer.getvalue(), flush=True)
        raise RuntimeError("Llama.cpp server failed to start or crashed.")

    time.sleep(check_interval)

if not server_ready:
    print("\nERROR CRÍTICO: El servidor llama.cpp no se inició correctamente o no se puso a escuchar a tiempo.", flush=True)
    print("\n--- Salida Final de LLAMA.cpp STDOUT ---\n" + llama_stdout_buffer.getvalue(), flush=True)
    print("\n--- Salida Final de LLAMA.cpp STDERR ---\n" + llama_stderr_buffer.getvalue(), flush=True)
    raise RuntimeError("Llama.cpp server failed to start within timeout.")


# 6. Configurar y lanzar localtunnel
print("\n--- Configurando Localtunnel ---", flush=True)

# Instalar localtunnel globalmente
print("Instalando localtunnel (verbose)...")
get_ipython().system('npm install -g localtunnel --force -ddd')

# Terminar cualquier instancia anterior de localtunnel
kill_process("lt --port 8889")

# Obtener la ruta del ejecutable de localtunnel
localtunnel_executable = "lt" # Fallback por defecto
try:
    npm_bin_path = subprocess.run(['npm', 'bin', '-g'], capture_output=True, text=True, check=True).stdout.strip()
    candidate_lt_path = os.path.join(npm_bin_path, 'lt')
    if os.path.exists(candidate_lt_path):
        localtunnel_executable = candidate_lt_path
        print(f"Ruta del ejecutable de localtunnel: {localtunnel_executable}", flush=True)
    else:
        print(f"ADVERTENCIA: El ejecutable '{candidate_lt_path}' no se encontró en la ruta esperada. Intentando 'lt' desde PATH.", flush=True)
except subprocess.CalledProcessError as e:
    print(f"Error al obtener la ruta de npm bin -g: {e}. Intentando 'lt' desde PATH.", flush=True)

# Verificar si localtunnel está en PATH o accesible (comprobación final)
lt_check = subprocess.run(['which', localtunnel_executable], capture_output=True, text=True)
if lt_check.returncode == 0:
    localtunnel_executable = lt_check.stdout.strip()
    print(f"'{localtunnel_executable}' encontrado en: {localtunnel_executable}", flush=True)
else:
    print(f"ERROR CRÍTICO: '{localtunnel_executable}' NO encontrado en PATH ni en la ruta esperada. La instalación de localtunnel pudo haber fallado.", flush=True)
    print(lt_check.stderr.strip(), flush=True)
    raise RuntimeError("Localtunnel executable not found.")


# Buffers para capturar la salida de localtunnel
lt_stdout_buffer = io.StringIO()
lt_stderr_buffer = io.StringIO()

print(f"Iniciando localtunnel directamente con subprocess.Popen.", flush=True)

lt_process = subprocess.Popen(
    [localtunnel_executable, '--port', '8889'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    bufsize=1,
    universal_newlines=False
)

# Hilos para leer stdout y stderr de localtunnel
lt_stdout_thread = threading.Thread(target=read_stream_in_thread, args=(lt_process.stdout, lt_stdout_buffer, 'LT_STDOUT'))
lt_stderr_thread = threading.Thread(target=read_stream_in_thread, args=(lt_process.stderr, lt_stderr_buffer, 'LT_STDERR'))
lt_stdout_thread.daemon = True
lt_stderr_thread.daemon = True
lt_stdout_thread.start()
lt_stderr_thread.start()

tunnel_url = None
localtunnel_timeout = 60
localtunnel_start_time = time.time()

print(f"Esperando a que localtunnel establezca la conexión (hasta {localtunnel_timeout} segundos)...")

while time.time() - localtunnel_start_time < localtunnel_timeout:
    current_lt_stdout_full = lt_stdout_buffer.getvalue()

    match = re.search(r"https:\/\/([a-zA-Z0-9-]+\.)*loca\.lt", current_lt_stdout_full)
    if match:
        tunnel_url = match.group(0)
        break

    if lt_process.poll() is not None: # El proceso ha terminado prematuramente
        print(f"\nERROR: El proceso de localtunnel terminó inesperadamente con código de salida {lt_process.returncode}.", flush=True)
        print("\n--- Salida Final de Localtunnel STDOUT ---\n" + lt_stdout_buffer.getvalue(), flush=True)
        print("\n--- Salida Final de Localtunnel STDERR ---\n" + lt_stderr_buffer.getvalue(), flush=True)
        raise RuntimeError("Localtunnel failed to start or crashed.")

    time.sleep(check_interval)

if tunnel_url:
    print(f"\n¡Localtunnel establecido con éxito! Tu URL del túnel es: {tunnel_url}", flush=True)
else:
    print(f"\n--- Tiempo de espera de localtunnel agotado ({localtunnel_timeout}s) ---", flush=True)
    print("No se pudo encontrar la URL de localtunnel dentro del tiempo dado.", flush=True)
    print("\n--- Salida Final de Localtunnel STDOUT ---\n" + lt_stdout_buffer.getvalue(), flush=True)
    print("\n--- Salida Final de Localtunnel STDERR ---\n" + lt_stderr_buffer.getvalue(), flush=True)

    # Comprobar si el proceso está en ejecución
    if lt_process.poll() is None:
        print("\nEl proceso de localtunnel todavía está en ejecución, pero no se encontró la URL.", flush=True)
        print("Puede que esté esperando alguna interacción o haya fallado silenciosamente.", flush=True)
    else:
        print("\nEl proceso de localtunnel NO está ejecutándose. Ha fallado al iniciar o se ha caído.", flush=True)

    print("\n--- Consejos para la resolución de problemas (Localtunnel) ---\n", flush=True)
    print("1. Revisa la salida de `npm install -g localtunnel` de arriba.", flush=True)
    print("2. Revisa la 'Salida de Error final de Localtunnel' para obtener pistas.", flush=True)
    print("3. Si el problema persiste, podría ser un problema temporal con el servicio de localtunnel.me.", flush=True)

print("\nFin de la ejecución del script principal.", flush=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelo ya presente en Drive.
Ejecutable encontrado en Drive. Copiando a entorno local...

--- Iniciando Servidor Llama.cpp ---
Iniciando llama-server directamente con subprocess.Popen.
Esperando a que el servidor llama.cpp esté listo (hasta 180 segundos)...


/usr/lib/python3.12/subprocess.py:1016: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdout = io.open(c2pread, 'rb', bufsize)
/usr/lib/python3.12/subprocess.py:1021: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stderr = io.open(errread, 'rb', bufsize)


[LLAMA_STDERR] 0.00.127.603 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
[LLAMA_STDERR] 0.00.312.780 I srv    load_model: loading model '/content/drive/MyDrive/Colab Notebooks/llm_setup/qwen2.5-7b-instruct-q4_k_m.gguf'
[LLAMA_STDERR] 0.01.460.590 W load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
[LLAMA_STDERR] 0.18.803.738 I srv    load_model: initializing, n_slots = 4, n_ctx_slot = 32768, kv_unified = 'true'
[LLAMA_STDERR] 0.18.821.897 I srv  llama_server: model loaded
[LLAMA_STDERR] 0.18.821.929 I srv  llama_server: listening on http://0.0.0.0:8889

Servidor llama.cpp listo y escuchando en el puerto 8889.

--- Configurando Localtunnel ---
Instalando localtunnel (verbose)...
npm verbose cli /tools/node/bin/node /tools/node/bin/npm
npm info using npm@10.8.2
npm info using node@v20.19.0
npm silly config load:file:/tools/node/lib/node_modules/npm/npmrc
npm s

/usr/lib/python3.12/subprocess.py:1016: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdout = io.open(c2pread, 'rb', bufsize)
/usr/lib/python3.12/subprocess.py:1021: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stderr = io.open(errread, 'rb', bufsize)


[LT_STDOUT] your url is: https://modern-paws-dress.loca.lt

¡Localtunnel establecido con éxito! Tu URL del túnel es: https://modern-paws-dress.loca.lt

Fin de la ejecución del script principal.
